# Step 3. Give every measure its category

Every measure from step 2 was read in full (its heading, its text, the boxes inside it and, for the measures that exist
only in a cost table, the table's one-line description) and given one of 15 categories by hand. The decisions are in
**`manual/categories.csv`**, one row per measure, each with a short reason. The rules are in
**`manual/category_guide.md`**:

1. A measure aimed only at Indigenous peoples goes to **Indigenous**.
2. Otherwise a measure goes where **what it is for** fits best, not how it is delivered (a tax cut to lower people's
   costs is Affordability; a tax write-off for business investment is Trade, industry & competitiveness).
3. **Taxation** and **Government** hold only the tax system and the running of government, plus the money they raise
   or save.
4. **Other** only when nothing else fits (foreign aid, for example).

A measure that undoes an earlier policy keeps the category of what it does, and the `reverses` column names the
category of the policy it undoes (the consumer carbon price cancellation, 2025: Affordability, reverses Clean energy &
climate).

This notebook only joins the two files and checks them: every measure has exactly one known category, no decision is
left over from a measure that no longer exists, and the cost copied into the decisions file still matches step 2's.

Output: `data/categorised.csv` (one row per measure, without its text) and `data/category_list.csv` (the 15
categories, their order and short names).

In [1]:
import pandas as pd

DATA, MANUAL = "data", "manual"

# the 15 categories, in a fixed order (the order is the id the chart page uses; Other is last)
CATEGORIES = [
    "Housing",
    "Affordability & social support",
    "Health care",
    "Trade, industry & competitiveness",
    "Jobs, skills & immigration",
    "Infrastructure",
    "Research, AI & technology",
    "Culture & communities",
    "Nature & environment",
    "Clean energy & climate",
    "Taxation",
    "Government",
    "Defence & security",
    "Indigenous",
    "Other",
]

## Read the measures (step 2) and the decisions (made by hand)

In [2]:
measures = pd.read_csv(f"{DATA}/measures.csv")                       # one row per measure, with its text
decisions = pd.read_csv(f"{MANUAL}/categories.csv")                  # one row per measure: category, reason, reverses
print(f"{len(measures):,} measures from step 2, {len(decisions):,} decisions in {MANUAL}/categories.csv")

1,317 measures from step 2, 1,317 decisions in manual/categories.csv


## Checks
- each measure has one decision, and each decision belongs to a measure (an id is year | page | heading, so a heading
  that changes on the budget site, or in step 2's reading, shows up here);
- every category and every `reverses` value is one of the 15;
- every decision has a reason;
- the cost copied into the decisions file (to read it on its own) still equals step 2's cost.

In [3]:
missing = sorted(set(measures["id"]) - set(decisions["id"]))         # measures nobody has decided on
extra = sorted(set(decisions["id"]) - set(measures["id"]))           # decisions whose measure is gone
assert decisions["id"].is_unique, "a measure is decided twice"
print("measures without a decision:", len(missing))
for i in missing[:20]:
    print("   ", i)
print("decisions without a measure:", len(extra))
for i in extra[:20]:
    print("   ", i)
assert not missing and not extra, "decisions and measures don't line up (see the lists above)"

bad = decisions[~decisions["category"].isin(CATEGORIES)]             # a misspelt or unknown category
assert bad.empty, f"unknown categories: {sorted(bad['category'].unique())}"
rev = decisions["reverses"].dropna()
assert rev.isin(CATEGORIES).all(), f"unknown reverses: {sorted(set(rev) - set(CATEGORIES))}"
assert decisions["reason"].notna().all(), "a decision has no reason"

# the cost in the decisions file is a copy, for reading; step 2's is the one used
both = measures[["id", "money_b"]].merge(decisions[["id", "money_b"]], on="id", suffixes=("", "_decisions"))
changed = both[(both["money_b"].fillna(-999) - both["money_b_decisions"].fillna(-999)).abs() > 0.0005]
print("measures whose cost changed since they were decided:", len(changed))
print(changed.head(20).to_string(index=False))
print("all checks passed")

measures without a decision: 0
decisions without a measure: 0
measures whose cost changed since they were decided: 0
Empty DataFrame
Columns: [id, money_b, money_b_decisions]
Index: []
all checks passed


## Join and save
One row per measure: where it is in the budget, its cost (net, $ billions over the table's five or six years; empty
when the cost tables have no row for it), its words, its category, the reason, and the policy category it reverses (if
any). The text stays in `data/measures.csv`.

In [4]:
out = measures.drop(columns=["text"]).merge(decisions[["id", "category", "reason", "reverses"]], on="id", how="left")
out["cat_id"] = out["category"].map({c: k for k, c in enumerate(CATEGORIES)})   # 0 to 14, in the order above
out.to_csv(f"{DATA}/categorised.csv", index=False)

cats = pd.DataFrame({"cat_id": range(len(CATEGORIES)), "category": CATEGORIES,
                     "short": [c.split(" & ")[0].split(", ")[0] for c in CATEGORIES]})
cats.to_csv(f"{DATA}/category_list.csv", index=False)
print(f"wrote {DATA}/categorised.csv ({len(out):,} rows) and {DATA}/category_list.csv ({len(cats)} categories)")

wrote data/categorised.csv (1,317 rows) and data/category_list.csv (15 categories)


## What the categories hold
Number of measures, net money ($ billions, five years) and words, by category and budget. Net money: new spending
minus savings and money raised within the category (the charts keep the two apart; this table is for checking).

In [5]:
pd.set_option("display.width", 200)
count = out.pivot_table(index="category", columns="year", values="id", aggfunc="count", fill_value=0).reindex(CATEGORIES)
count["all"] = count.sum(axis=1)
print("Measures\n", count.to_string(), "\n")

money = out.pivot_table(index="category", columns="year", values="money_b", aggfunc="sum", fill_value=0).reindex(CATEGORIES)
money.loc["Total"] = money.sum()
print("Net money, $ billions\n", money.round(1).to_string(), "\n")

words = out.pivot_table(index="category", columns="year", values="words", aggfunc="sum", fill_value=0).reindex(CATEGORIES)
words.loc["Total"] = words.sum()
print("Words\n", words.to_string())

Measures
 year                               2021  2022  2023  2024  2025  2026  all
category                                                                  
Housing                               4    22     5    32     3     7   73
Affordability & social support       11     6    11    23    10     9   70
Health care                          24    15    15    15     2     7   78
Trade, industry & competitiveness    33    16    19    26    25    27  146
Jobs, skills & immigration           38    19    23    28    14    11  133
Infrastructure                       10     5     7     7     5     9   43
Research, AI & technology            16    12     5    12     5     6   56
Culture & communities                25    16    15    26    10    14  106
Nature & environment                 14     6     8     9     1     5   43
Clean energy & climate               31    22    16    18     8    10  105
Taxation                             13    14    10     9     3    10   59
Government     

## Measures that reverse an earlier policy

In [6]:
flag = out[out["reverses"].notna()]
print(flag[["year", "heading", "money_b", "category", "reverses"]].to_string(index=False))

 year                                       heading  money_b                       category               reverses
 2025         Clarity on Greenhouse Gas Regulations      NaN         Clean energy & climate Clean energy & climate
 2025 Cancelling the Divisive Consumer Carbon Price    4.185 Affordability & social support Clean energy & climate


## The largest measures of each budget, with their category
A quick way to check the decisions that move the most money.

In [7]:
for year, part in out.groupby("year"):
    top = part.reindex(part["money_b"].abs().sort_values(ascending=False).index).head(8)
    print(f"Budget {year}")
    for _, m in top.iterrows():
        print(f"   {m['money_b']:8.2f}  {m['category']:<34} {m['heading'][:80]}")

Budget 2021
      23.94  Affordability & social support     Establishing a Canada-Wide Early Learning and Child Care System
      13.98  Jobs, skills & immigration         Providing Additional Weeks of Recovery Benefits and EI Regular Benefits
      10.67  Affordability & social support     Increasing Old Age Security for Canadians 75 and Over
      10.14  Jobs, skills & immigration         Extending the Canada Emergency Wage Subsidy
       7.17  Affordability & social support     Enhancing the Canada Workers Benefit
       5.94  Infrastructure                     Permanent Public Transit Fund
       5.76  Jobs, skills & immigration         Maintaining Flexible Access to Employment Insurance Benefits
      -5.32  Taxation                           Limitations on Excessive Interest Deductions
Budget 2022
      -6.11  Taxation                           Requiring Financial Institutions to Help Pay for the Recovery
       6.10  Defence & security                 Reinforcing our Defence Pri